# AgriCL-Net, step 1: lesion segmentation (SAM2-UNet on PlantSeg)

Trains a binary **lesion vs. background** model. All PlantSeg diseases share one "lesion" class, so the model
also works on diseases it has never seen. Three diseases (soybean bacterial blight, wheat stripe rust,
apple rust) are **held out of training** and evaluated separately: that is the project's key experiment.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. The project code is cloned from GitHub;
nothing needs uploading. Run the cells in order. The first run is a **smoke test** (40 images, 1 epoch).
When it finishes without errors, set `SMOKE_TEST = False` and choose Runtime > Run all.

Everything slow (dataset zip, SAM2 checkpoint, training checkpoints) is saved to Google Drive, so a
disconnect only loses the current epoch: run all cells again and training resumes.

## Settings

In [ ]:
SIZE = 'l'          # Hiera backbone: 't' (fastest), 's', 'b+', 'l' (most accurate, the SAM2-UNet default)
IMG_SIZE = 352      # SAM2-UNet default; must be a multiple of 32
BATCH = 4           # lower to 2 if the model check reports out-of-memory
EPOCHS = 20         # SAM2-UNet default
SMOKE_TEST = True   # True: 40 images, 1 epoch. Set False for the real run.

## GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Storage and project code from GitHub

Works on **Colab** and **Kaggle**:
* Colab: everything is kept in Google Drive, `MyDrive/Colab Notebooks/agricl-net`.
* Kaggle: turn on *Settings > Accelerator > GPU T4* and *Settings > Internet*. Results go to
  `/kaggle/working/agricl-net` and are kept when you *Save Version*. Downloads go to `/tmp` (fetched again each
  session). To use an earlier notebook's results (or resume a run), attach that notebook's output with
  *Add Input > Notebook Output*: it is copied in automatically below.

In [ ]:
import os, sys, glob, shutil
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    WORK = '/tmp'                               # scratch space: code, unzipped data
    DRIVE = '/kaggle/working/agricl-net'        # results, saved with each notebook version
    CACHE = '/tmp/agricl-cache'                 # dataset zip and SAM2 checkpoint
    for src in glob.glob('/kaggle/input/**/agricl-net/runs/*', recursive=True):
        dst = f'{DRIVE}/runs/{os.path.basename(src)}'
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print('copied earlier results:', os.path.basename(src))
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content'
    DRIVE = '/content/drive/MyDrive/Colab Notebooks/agricl-net'   # dataset zip, SAM2 checkpoint and all results
    CACHE = f'{DRIVE}/cache'
os.makedirs(CACHE, exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)

CODE = f'{WORK}/AGRICL-NET'
if os.path.exists(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q https://github.com/Dev02005/AGRICL-NET.git {CODE}
sys.path.insert(0, CODE)
!git -C {CODE} log -1 --format='code version: %h %s'

## PlantSeg v7 dataset (1.1 GB; on Colab downloaded to Drive once)

In [ ]:
import hashlib
ZIP = f'{CACHE}/plantseg.zip'
if not os.path.exists(ZIP):
    !wget -q --show-progress -O "{ZIP}" "https://zenodo.org/records/17719108/files/plantseg.zip?download=1"

h = hashlib.md5()
with open(ZIP, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 24), b''):
        h.update(chunk)
assert h.hexdigest() == '9358a66dff88cdd15c4fe009763c40a3', f'bad download, md5 {h.hexdigest()}: delete {ZIP} and rerun'

DATA = f'{WORK}/data/plantseg'
if not os.path.exists(f'{DATA}/Metadata.csv'):
    !unzip -q "{ZIP}" -d "{WORK}/data"

## SAM2-UNet code and SAM2 checkpoint

In [ ]:
!pip -q install hydra-core==1.3.2
SAM2UNET = f'{WORK}/SAM2-UNet'
if not os.path.exists(SAM2UNET):
    !git clone -q https://github.com/WZH0120/SAM2-UNet.git {SAM2UNET}
    !git -C {SAM2UNET} checkout -q 01598e5e9912ffb23f965ecbebf4d1dfecbaa56e

from agricl.models.sam2unet import CHECKPOINTS, CHECKPOINT_URL
CKPT = f'{CACHE}/{CHECKPOINTS[SIZE]}'
if not os.path.exists(CKPT):
    !wget -q --show-progress -O "{CKPT}" "{CHECKPOINT_URL}{CHECKPOINTS[SIZE]}"
print(CKPT, os.path.getsize(CKPT) // 2**20, 'MB')

In [ ]:
RUN = f"lesion_sam2unet_{SIZE.replace('+', 'p')}_{IMG_SIZE}" + ('_smoke' if SMOKE_TEST else '')
OUT = f'{DRIVE}/runs/{RUN}'
print('results ->', OUT)

## Model check: one training step on random data (shapes and GPU memory)

In [ ]:
import torch
from agricl.models.sam2unet import LesionSAM2UNet, structure_loss

m = LesionSAM2UNet(SAM2UNET, SIZE, CKPT).cuda().train()
x = torch.randn(BATCH, 3, IMG_SIZE, IMG_SIZE, device='cuda')
y = (torch.rand(BATCH, 1, IMG_SIZE, IMG_SIZE, device='cuda') > 0.8).float()
with torch.autocast('cuda', dtype=torch.float16):
    outs = m(x)
loss = sum(structure_loss(o.float(), y) for o in outs)
loss.backward()
print('outputs:', [tuple(o.shape) for o in outs], 'loss:', round(loss.item(), 4))
print(f'peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.1f} GB')
del m, x, y, outs, loss
torch.cuda.empty_cache()

## Train (resumes automatically from Drive)

In [ ]:
from agricl.train_lesion import TrainConfig, train, final_report

cfg = TrainConfig(
    data_root=DATA, out_dir=OUT, repo_dir=SAM2UNET,
    size=SIZE, checkpoint=CKPT, img_size=IMG_SIZE, batch_size=BATCH,
    epochs=1 if SMOKE_TEST else EPOCHS, limit=40 if SMOKE_TEST else 0, workers=2,
)
model = train(cfg)

## Final evaluation at full resolution

* `test_seen`: PlantSeg test images of the diseases used in training.
* `unseen`: all images of the three held-out diseases. If these scores are close to `test_seen`,
  the lesion model works on diseases it has never seen.
* `share_r2` / `share_rmse`: predicted vs. true lesion share of the whole image.

In [ ]:
results = final_report(cfg, model)

## Training curve and example predictions

In [ ]:
import json, matplotlib.pyplot as plt, numpy as np
from agricl.data.plantseg import load_image, load_lesion_mask
from agricl.train_lesion import build_splits, predict_mask

hist = json.load(open(f'{OUT}/history.json'))
ep = [h['epoch'] for h in hist]
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(ep, [h['train_loss'] for h in hist]); ax[0].set_title('train loss'); ax[0].set_xlabel('epoch')
ax[1].plot(ep, [h['val_mean_dice'] for h in hist]); ax[1].set_title('val mean Dice'); ax[1].set_xlabel('epoch')
plt.show()

samples = build_splits(cfg)['unseen'][:4]
fig, ax = plt.subplots(len(samples), 3, figsize=(9, 3 * len(samples)))
for row, r in zip(np.atleast_2d(ax), samples):
    img = load_image(r)
    row[0].imshow(img); row[0].set_title(r.disease)
    row[1].imshow(load_lesion_mask(r), cmap='gray'); row[1].set_title('true lesion')
    row[2].imshow(predict_mask(model, img, IMG_SIZE, 'cuda'), cmap='gray'); row[2].set_title('predicted')
    for a in row: a.axis('off')
plt.tight_layout(); plt.show()

## Next
* Smoke test passed? Set `SMOKE_TEST = False` in the settings cell and run all cells.
* Results: `MyDrive/Colab Notebooks/agricl-net/runs/<run name>/` (`best.pt`, `history.json`, `results.json`, per-image CSVs).
* If one epoch takes too long for a Colab session, try `SIZE = 'b+'` or `'s'` and compare the scores.
* When the full run is done, open `02_leaf_and_severity.ipynb`.